# Day 4 · Streaming and quality reference

Executed source reference only. No Kafka, Spark, Delta or Great Expectations execution is claimed.



[Day 4](README.md) · [Sources](SOURCES.md)



## 1. Setup and source integrity



In [ ]:
from pathlib import Path
import sys, json
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the complete course repository")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.workspace import require_fixed_dataset
require_fixed_dataset(SOURCE)
print("MASAR_SMALL_V1: original source hashes verified")

## 2. Derive the Day 4 reference



In [ ]:
from masar.trust_reference import day04_reference
reference = day04_reference(SOURCE)
print(json.dumps({k: reference[k] for k in ["scope", "engine_executed", "kafka_executed", "gx_executed", "deliveries", "distinct_events"]}, indent=2))

## 3. Explain messages versus events



In [ ]:
from masar.trust_reference import events_from_source, unique_events, GPS_FILES
all_events = []
for filename in GPS_FILES:
    all_events.extend(events_from_source(SOURCE, filename))
    print(f"{filename}: source deliveries={len(all_events)}, distinct event IDs={len(unique_events(all_events))}")
print("These are source-derived counts, not observed Kafka offsets or checkpoint recovery.")

## 4. Apply the explicit quality policy



In [ ]:
from masar.delta_reference import day03_reference
from masar.silver_reference import read_csv, drivers_index
from masar.trust_reference import quality_probe_rows, evaluate_quality
trusted = day03_reference(SOURCE)["expected_corrected_rows"]
driver_ids = set(drivers_index(read_csv(SOURCE, "drivers.csv")))
clean = evaluate_quality(trusted, driver_ids)
mixed = evaluate_quality(trusted + quality_probe_rows(SOURCE), driver_ids)
rechecked = evaluate_quality(mixed["accepted"], driver_ids)
for label, result in [("trusted", clean), ("mixed", mixed), ("rechecked", rechecked)]:
    print(label, {k: result[k] for k in ["input_rows", "accepted_rows", "rejected_rows", "promote_allowed"]})
print("Policy execution is not a Great Expectations run.")

## 5. Read all seven root reasons



In [ ]:
for rejected in mixed["quarantine"]:
    print(rejected["candidate_row"], rejected["trip_id"] or "<missing key>", ", ".join(rejected["reason_codes"]))
print(f"Rejected: {mixed['rejected_rows']} / {mixed['input_rows']} = {mixed['rejection_rate']:.4%}")
assert mixed["input_rows"] == mixed["accepted_rows"] + mixed["rejected_rows"]

## 6. Use an explicit historical clock



In [ ]:
print("As of:", reference["scenario_clock"])
print("Newest event:", reference["event_time_newest"])
print("Source event freshness:", reference["source_event_freshness"])
print("Delivery freshness:", reference["delivery_freshness"])
from masar.trust_reference import freshness
future_probe = freshness("2026-06-04T03:06:00Z", "2026-06-04T03:05:00Z", max_age_seconds=300)
assert future_probe["status"] == "FUTURE_TIMESTAMP"
print("A future timestamp is not automatically fresh:", future_probe)

## 7. Interpret a distribution warning



In [ ]:
from copy import deepcopy
from masar.trust_reference import city_distribution
print("Accepted rows against Day 3:", city_distribution(rechecked["accepted"], trusted))
probe = deepcopy(trusted)
for row in probe:
    row["city"] = "Riyadh"
print("Transient example only, not a source edit:", city_distribution(probe, trusted))
assert rechecked["accepted"] == trusted

## 8. Save the bounded evidence



In [ ]:
from masar.workspace import new_workspace, workspace_path, write_json
assert all(reference["checks"].values())
assert reference["engine_executed"] is reference["kafka_executed"] is reference["gx_executed"] is False
work = new_workspace(ROOT, "day04_reference")
evidence = workspace_path(work, "reports/day04_reference.json")
write_json(evidence, reference)
print("Reference checks:", len(reference["checks"]), "passed")
print("Preserved trusted amount:", reference["trusted_fare_sar"], "SAR")
print("Saved under a new outputs/day04_reference_* workspace.")
print("Native Kafka/Delta/GX, checkpoints and Data Docs still require actual execution.")